# Tree Structures

The `dags.tree` module provides utilities for working with nested dictionary structures, converting between tree representations and flat qualified names.

## Overview

When working with hierarchical data or organizing functions into namespaces, you often need to convert between:

- **Nested dictionaries** (tree structure): `{"a": {"b": 1, "c": 2}}`
- **Qualified names** (flat strings): `{"a__b": 1, "a__c": 2}`
- **Tree paths** (flat tuples): `{("a", "b"): 1, ("a", "c"): 2}`

The tree module lets you **organize functions into namespaces** (like `"taxes"`, `"income"`, `"transfers"`) and handles the conversion between nested and flat representations automatically. This keeps your code organized while still working with dags' flat function dictionaries under the hood.

In [ ]:
import dags.tree as dt

## Qualified Names

Qualified names (qnames) are flat strings that encode hierarchy using a double-underscore delimiter. For example, `"household__income"` represents the `"income"` entry inside the `"household"` namespace. dags uses this convention throughout to bridge between nested and flat representations.

In [ ]:
dt.QNAME_DELIMITER

In [ ]:
qname = dt.qname_from_tree_path(("household", "income"))
qname

In [ ]:
path = dt.tree_path_from_qname("household__income")
path

## Flattening and Unflattening

These are the core operations of the tree module. **Flattening** converts a nested dictionary into a flat one (for use with `concatenate_functions`), and **unflattening** restores the nested structure (for readable output).

### To/From Qualified Names

Let's define a nested dictionary representing a household budget:

In [ ]:
tree = {
    "household": {
        "income": 50000,
        "expenses": 30000,
    },
    "taxes": {
        "federal": 10000,
        "state": 2500,
    },
}

In [ ]:
flat = dt.flatten_to_qnames(tree)
flat

In [ ]:
dt.unflatten_from_qnames(flat)

### To/From Tree Paths

In [ ]:
flat_paths = dt.flatten_to_tree_paths(tree)
flat_paths

In [ ]:
dt.unflatten_from_tree_paths(flat_paths)

## Extracting Names and Paths

Sometimes you just need the keys of a nested structure as a flat list — for example, to specify targets or to inspect what a tree contains:

In [ ]:
small_tree = {
    "a": {
        "x": 1,
        "y": 2,
    },
    "b": 3,
}

dt.qnames(small_tree)

In [ ]:
dt.tree_paths(small_tree)

## Tree DAG Functions

So far we've used the tree module for data manipulation. But its real power is working with **nested function dictionaries** — organizing your model's functions into namespaces, then letting dags handle the flattening automatically.

### create_dag_tree

`create_dag_tree` builds a dependency graph directly from nested function dictionaries, without requiring you to flatten manually:

In [ ]:
functions = {
    "inputs": {
        "income": lambda: 50000,
        "tax_rate": lambda: 0.3,
    },
    "calculations": {
        "tax": lambda inputs__income, inputs__tax_rate: (
            inputs__income * inputs__tax_rate
        ),
        "net": lambda inputs__income, calculations__tax: (
            inputs__income - calculations__tax
        ),
    },
}

dag = dt.create_dag_tree(
    functions=functions,
    inputs={},
    targets={"calculations": {"net": None}},
)

list(dag.nodes())

### concatenate_functions_tree

This is the tree-aware equivalent of `concatenate_functions`. It takes nested function and target dictionaries, and returns a combined function whose **inputs and outputs are also nested dictionaries**:

In [ ]:
combined = dt.concatenate_functions_tree(
    functions=functions,
    inputs={},
    targets={"calculations": {"tax": None, "net": None}},
)

combined({})

### functions_without_tree_logic

If you want to use the regular `concatenate_functions` but start with nested function dictionaries, this utility flattens the functions and converts their parameter names to qualified names:

In [ ]:
tree_functions = {
    "a": {
        "x": lambda: 1,
        "y": lambda a__x: a__x + 1,
    },
}

flat_functions = dt.functions_without_tree_logic(
    tree_functions, top_level_namespace={"a"}
)

list(flat_functions.keys())

### create_tree_with_input_types

Discover the expected input structure (with type annotations) for a set of tree
functions and targets. This is useful when you want to know what inputs a tree DAG
expects before calling it:

In [ ]:
typed_functions = {
    "income": {
        "wage": lambda hours, hourly_wage: hours * hourly_wage,
        "capital": lambda wealth, interest_rate: wealth * interest_rate,
    },
    "taxes": {
        "income_tax": lambda income__wage, income__capital: (
            0.3 * (income__wage + income__capital)
        ),
    },
}

input_types = dt.create_tree_with_input_types(
    functions=typed_functions,
    targets={"taxes": {"income_tax": None}},
)
input_types

### one_function_without_tree_logic

Convert a single function's parameters from relative names to qualified absolute names.
This is the building block used by `functions_without_tree_logic`:

In [ ]:
import dags


def my_func(wage, capital):
    return wage + capital


converted = dt.one_function_without_tree_logic(
    function=my_func,
    tree_path=("taxes", "income_tax"),
    top_level_namespace={"income", "taxes"},
)

dags.get_free_arguments(converted)

## Validation Functions

The tree module includes utilities to catch common mistakes early — before they cause confusing errors downstream.

Trailing underscores in non-leaf path elements are not allowed:

In [ ]:
dt.fail_if_path_elements_have_trailing_underscores({("bad_", "name")})

Top-level namespace elements must not be repeated further down in the hierarchy:

In [ ]:
dt.fail_if_top_level_elements_repeated_in_paths(
    all_tree_paths={("a", "b", "a")},
    top_level_namespace={"a", "b"},
)

## Real-World Example

Here's how you might use tree utilities for regime management — defining similar
functions for different regimes, then combining them.

Functions within a namespace can reference each other using relative names. For example,
`utility` in the `"working"` namespace uses `income` (resolved to `working__income`) and
`leisure` (resolved to `working__leisure` — but since `"leisure"` is a top-level input,
it stays as `leisure`).

In [ ]:
regime_functions = {
    "working": {
        "income": lambda wage, hours: wage * hours,
        "utility": lambda income, leisure: income + leisure,
    },
    "retired": {
        "income": lambda pension: pension,
        "utility": lambda income, leisure: income + 2 * leisure,
    },
}

In [ ]:
flat_regime_functions = dt.functions_without_tree_logic(
    regime_functions,
    top_level_namespace={"working", "retired", "wage", "hours", "pension", "leisure"},
)

combined = dags.concatenate_functions(
    functions=flat_regime_functions,
    targets=["working__utility", "retired__utility"],
    return_type="dict",
)

result = combined(wage=20, hours=40, pension=1000, leisure=10)
dt.unflatten_from_qnames(result)

Alternatively, use `concatenate_functions_tree` to handle the flatten/unflatten
automatically:

In [ ]:
input_structure = {
    "wage": None,
    "hours": None,
    "pension": None,
    "leisure": None,
}

combined_tree = dt.concatenate_functions_tree(
    functions=regime_functions,
    inputs=input_structure,
    targets={"working": {"utility": None}, "retired": {"utility": None}},
)

combined_tree(
    {
        "wage": 20,
        "hours": 40,
        "pension": 1000,
        "leisure": 10,
    }
)